# Jeff stock replication — Colab worker 1

Use this notebook and the other worker in your two separate **fresh GPU runtimes**. Each reproduces its historical shard, totaling 400 target identities. The runner reconstructs the seven recorded source runs and their resume boundaries; batches may include companion episodes needed to preserve historical membership. Add Supabase Secrets and Run all. Stop any previous sweep in that runtime first.

This is **stock SmolVLA only: no P&P, no re-noising, no critic and no reranking**. It uses Jeff's stream-0 episode seeds, ten Euler steps, 50 generated actions, ten executed actions, historical same-task batch limits (two or eight), historical rendering settings, matmul TF32 disabled and highest precision. cuDNN TF32 retains its Torch default because its historical value is unknown. Torch is pinned to Jeff's 2.11.0+cu128 build. Your Colab GPU may differ from Jeff's L4; outcome agreement is checked rather than assumed.

Results use a new v2 experiment; v1 rows are never reused. Resume skips whole reconstructed batches, preserving companion lanes rather than shrinking partially completed batches. At the end each notebook reads the latest combined results and compares every episode against Jeff's saved 255/400 stock arm. It reports successes, outcome flips, seed mismatches and step-count mismatches. A partial report is expected if the other worker is still running. Equal aggregate totals alone do not establish replication. No PCP sweep starts from this notebook.


In [ ]:
WORKER_INDEX = 1
BATCH_SIZE = 8
SIMULATOR_PROCESSES = 1  # parity runner uses Jeff's direct environment stepping
BUNDLE_PATH = ''
BUNDLE_BUCKET = 'artifacts'
BUNDLE_SHA256 = '671fe1c691fa072589462378c857e03c808f667c7e48f1fa189351e0252d37d4'
BUNDLE_STORAGE_KEY = 'smolvla_pcp_sweep/bundles/671fe1c691fa072589462378c857e03c808f667c7e48f1fa189351e0252d37d4/trial_bundle.zip'
CHECKPOINT_SHA256 = {'frozen_mc': '9b8800bc9f27116afcb1812f93f8a9f6767ac012c7cdac790502032fbb7d0c8e', 'cnn_mc': 'afcf832d43ffca44af843e4e5301fa0a84b1b2e485703e2a1dc13aa02b260151', 'td': '1e3fa3f093a711d0651b69e7f152a5b56a7e7a85ec754c45b0afe5834d9f324e'}


In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path
os.environ.update(MUJOCO_GL='egl', NVIDIA_DRIVER_CAPABILITIES='compute,utility,graphics',
                  TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')
from google.colab import userdata
for key in ('SUPABASE_URL','SUPABASE_SERVICE_KEY','HF_TOKEN'):
    try:
        value=userdata.get(key)
        if value: os.environ[key]=value
    except Exception: pass
for key in ('SUPABASE_URL','SUPABASE_SERVICE_KEY'):
    if not os.environ.get(key): raise RuntimeError(f'Add {key} in Colab Secrets and allow notebook access.')
if not BUNDLE_PATH:
    subprocess.run([sys.executable,'-m','pip','install','-q','supabase'],check=True)
    from supabase import create_client
    import hashlib
    client=create_client(os.environ['SUPABASE_URL'],os.environ['SUPABASE_SERVICE_KEY'])
    cached=Path('/content')/f'pcp_bundle_{BUNDLE_SHA256}.zip'
    if not cached.exists() or hashlib.sha256(cached.read_bytes()).hexdigest()!=BUNDLE_SHA256:
        data=client.storage.from_(BUNDLE_BUCKET).download(BUNDLE_STORAGE_KEY)
        if hashlib.sha256(data).hexdigest()!=BUNDLE_SHA256:
            raise RuntimeError('Downloaded bundle SHA256 mismatch')
        cached.write_bytes(data)
    BUNDLE_PATH=str(cached)
if __import__('hashlib').sha256(Path(BUNDLE_PATH).read_bytes()).hexdigest()!=BUNDLE_SHA256:
    raise RuntimeError('Bundle differs from the exact version pinned in this notebook')
PACKAGE=Path('/content/smolvla-pcp-trial')
PACKAGE.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in archive.namelist()):
        raise ValueError('Unsafe bundle path')
    archive.extractall(PACKAGE)
print({'python':sys.version,'executable':sys.executable},flush=True)
subprocess.run([sys.executable,'-m','pip','--version'],check=True)
install=subprocess.run([sys.executable,'-m','pip','install','-e',str(PACKAGE)+'[sim]',
                        'torch==2.11.0+cu128','torchvision==0.26.0+cu128',
                        '--extra-index-url','https://download.pytorch.org/whl/cu128'],
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
install_log=Path('/content/pcp_dependency_install.log')
install_log.write_text(install.stdout)
if install.returncode:
    print('\n'.join(install.stdout.splitlines()[-120:]),flush=True)
    raise RuntimeError(f'Dependency installation failed (exit {install.returncode}). '
                       f'Full log: {install_log}. Paste the ERROR lines above; no experiment has started.')
print('Simulation/model dependencies installed.',flush=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=False,stdout=subprocess.DEVNULL)
if str(PACKAGE) not in sys.path: sys.path.insert(0,str(PACKAGE))
subprocess.run([sys.executable,'-m','pip','install',
                'torch==2.11.0+cu128','torchvision==0.26.0+cu128',
                '--index-url','https://download.pytorch.org/whl/cu128'],check=True)
if 'torch' in sys.modules and sys.modules['torch'].__version__ != '2.11.0+cu128':
    raise RuntimeError('Torch was already imported. Restart the runtime and rerun with the pinned version.')
from pnp import env_setup
if env_setup._find_nvidia_egl_library() is None:
    import re
    driver=subprocess.check_output(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],text=True).splitlines()[0]
    major=re.match(r'\d+',driver.strip()).group()
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','--no-install-recommends',f'libnvidia-gl-{major}'],check=True)
env_setup._ensure_nvidia_egl_vendor()
if env_setup._find_nvidia_egl_library() is None: raise RuntimeError('NVIDIA EGL missing; hardware rendering is required.')
env_setup._ensure_libero_config()
env_setup._fix_torch_quant_compat()
env_setup._remove_broken_optional_torchaudio()
env_setup._ensure_libero_assets(token=os.getenv('HF_TOKEN'))
import torch, hashlib, json
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime.')
metadata=json.loads((PACKAGE/'bundle_manifest.json').read_text())
if metadata['checkpoints']!=CHECKPOINT_SHA256: raise RuntimeError('Unexpected checkpoint manifest')
CHECKPOINT_PATHS={name:str(PACKAGE/'checkpoints'/f'{name}.pt') for name in metadata['checkpoints']}
for name,path in CHECKPOINT_PATHS.items():
    if hashlib.sha256(Path(path).read_bytes()).hexdigest()!=metadata['checkpoints'][name]:
        raise RuntimeError(f'Checkpoint mismatch: {name}')
print({'gpu':torch.cuda.get_device_name(0),'batch_size':BATCH_SIZE,'worker':WORKER_INDEX,'simulator_processes':SIMULATOR_PROCESSES})


In [ ]:
import torch
if torch.__version__ != '2.11.0+cu128':
    raise RuntimeError('Restart the runtime so the pinned Torch build loads.')
from pnp.smolvla_jeff_replication import run_replication
report = run_replication(shard_index=WORKER_INDEX)
print(json.dumps({k:v for k,v in report.items() if k!='outcome_flips'},indent=2))
if report['outcome_flips']:
    print('Episode outcome differences:',json.dumps(report['outcome_flips'],indent=2))


Rerun the next cell after both workers finish to refresh the combined comparison.

In [ ]:
from pnp.smolvla_jeff_replication import comparison
report = comparison()
print(json.dumps(report,indent=2))
